# 01 — Perfilagem dos Dados (Data Profiling)

**Objetivo:** inventariar os 9 arquivos de `data/raw/` — tamanhos, tipos, chaves, duplicatas, nulos e integridade referencial — e preencher `docs/data_quality_report.md`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path('../data/raw')

FILES = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'translation': 'product_category_name_translation.csv',
}

# utf-8-sig lida com o BOM do arquivo de tradução
dfs = {name: pd.read_csv(RAW / fname, encoding='utf-8-sig') for name, fname in FILES.items()}

## 1. Visão geral

In [ ]:
overview = pd.DataFrame({
    name: {
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'duplicatas_totais': int(df.duplicated().sum()),
        '% nulos (máx. coluna)': round(df.isna().mean().max() * 100, 2),
    }
    for name, df in dfs.items()
}).T
overview

## 2. Tipos e nulos por tabela

In [ ]:
for name, df in dfs.items():
    print(f'=== {name} ===')
    display(df.dtypes.to_frame('dtype').join(df.isna().sum().to_frame('nulos')))
    print()

## 3. Chaves primárias — unicidade

In [ ]:
KEYS = {
    'customers': ['customer_id'],
    'geolocation': ['geolocation_zip_code_prefix'],  # não é única por linha
    'orders': ['order_id'],
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'order_reviews': ['review_id'],
    'products': ['product_id'],
    'sellers': ['seller_id'],
    'translation': ['product_category_name'],
}

pd.DataFrame({
    name: {
        'chave': ' + '.join(keys),
        'linhas': len(dfs[name]),
        'chaves_únicas': dfs[name].groupby(keys, dropna=False).ngroups,
    }
    for name, keys in KEYS.items()
}).T

## 4. Integridade referencial

In [ ]:
def check_fk(child_df, child_col, parent_df, parent_col, label):
    orphans = (~child_df[child_col].dropna().isin(parent_df[parent_col])).sum()
    print(f'{label}: {orphans} órfãos')
    return orphans

check_fk(dfs['orders'], 'customer_id', dfs['customers'], 'customer_id', 'orders → customers')
check_fk(dfs['order_items'], 'product_id', dfs['products'], 'product_id', 'order_items → products')
check_fk(dfs['order_items'], 'seller_id', dfs['sellers'], 'seller_id', 'order_items → sellers')
check_fk(dfs['order_items'], 'order_id', dfs['orders'], 'order_id', 'order_items → orders')
check_fk(dfs['order_payments'], 'order_id', dfs['orders'], 'order_id', 'order_payments → orders')
check_fk(dfs['order_reviews'], 'order_id', dfs['orders'], 'order_id', 'order_reviews → orders')
check_fk(dfs['products'], 'product_category_name', dfs['translation'], 'product_category_name', 'products → translation')

## 5. Checagens de domínio

In [ ]:
print('order_status:')
display(dfs['orders']['order_status'].value_counts())

print('payment_type:')
display(dfs['order_payments']['payment_type'].value_counts())

print('review_score fora de 1-5:', (~dfs['order_reviews']['review_score'].between(1, 5)).sum())
print('preço negativo ou zerado:', (dfs['order_items']['price'] <= 0).sum())
print('frete negativo:', (dfs['order_items']['freight_value'] < 0).sum())
print('\nPeríodo dos pedidos:')
ts = pd.to_datetime(dfs['orders']['order_purchase_timestamp'])
print(f'{ts.min()}  →  {ts.max()}')

## Próximos passos

1. Transferir os resultados para `docs/data_quality_report.md`.
2. Aplicar as decisões de limpeza em `src/clean_data.py`.
3. Validar com `pytest tests/test_data_quality.py`.